In [185]:
import refinitiv.data as rd
import pandas as pd
import numpy as np

rd.open_session()

<refinitiv.data.session.Definition object at 0x126005a90 {name='workspace'}>

In [186]:
STOCK = "NESTE.HE"
PARAMS = {"SDate": "2024-01-01", "EDate": "2025-12-31", "Frq": "D", "Curn": "EUR"}

In [ ]:
df = rd.get_data(
    universe=[STOCK],
    fields=[
        "TR.PriceClose.date",
        "TR.PriceClose",
        "TR.CompanyMarketCap",
        "TR.SharesOutstanding",
        "TR.PE",
        "TR.EPSActValue",
        "TR.EPSFRActValue",
        "TR.F.NetIncAfterTax",
        "TR.EPSActValue(Period=LTM)",
    ],
    parameters=PARAMS
)

if len(df.columns) == 10:
    df.columns = ["Instrument", "Date", "Price", "MktCap", "Shares", "REF_PE", "EPSAct", "EPSfr", "NetIncome", "EPSAct_LTM"]
    df = df.drop(columns="Instrument")
else:
    df.columns = ["Date", "Price", "MktCap", "Shares", "REF_PE", "EPSAct", "EPSfr", "NetIncome", "EPSAct_LTM"]

df["Date"] = pd.to_datetime(df["Date"])
for c in ["Price", "MktCap", "Shares", "REF_PE", "EPSAct", "EPSfr", "NetIncome", "EPSAct_LTM"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print(f"{len(df)} riviä")
df.head()

501 riviä


/opt/anaconda3/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


,Date,Price,MktCap,Shares,REF_PE,EPSAct,EPSfr,NetIncome,EPSAct_LTM
0,2024-01-02,32.4800,24983975163.8400,768199747,16.1272,3.0400,2.4600,1891000000,3.0670
1,2024-01-03,31.8000,24460911644.4000,768199747,15.7896,3.0400,2.4600,1891000000,3.0670
2,2024-01-04,32.2700,24822440841.6600,768199747,16.0229,3.0400,2.4600,1891000000,3.0670
3,2024-01-05,32.4000,24922438279.1999,768199747,16.0875,3.0400,2.4600,1891000000,3.0670
4,2024-01-08,32.2700,24822440841.6599,768199747,16.0229,3.0400,2.4600,1891000000,3.0670


In [188]:
# Itselasketut PE ja EP
df["PE_from_EPSAct"] = df["Price"] / df["EPSAct"]
df["EP_from_EPSAct"] = df["EPSAct"] / df["Price"]

df["PE_from_EPSfr"] = df["Price"] / df["EPSfr"]
df["EP_from_EPSfr"] = df["EPSfr"] / df["Price"]

df["EP_NetIncome_MktCap"] = df["NetIncome"] / df["MktCap"]
df["PE_from_NetIncome_MktCap"] = 1 / df["EP_NetIncome_MktCap"]

# Forward-fill
df["EPSAct_ff"] = df["EPSAct"].ffill()
df["EPSfr_ff"] = df["EPSfr"].ffill()
df["NI_ff"] = df["NetIncome"].ffill()
df["Shares_ff"] = df["Shares"].ffill()
df["EPS_from_NI"] = df["NI_ff"] / df["Shares_ff"]

df["PE_from_EPSAct_ff"] = df["Price"] / df["EPSAct_ff"]
df["EP_from_EPSAct_ff"] = df["EPSAct_ff"] / df["Price"]
df["PE_from_EPSfr_ff"] = df["Price"] / df["EPSfr_ff"]
df["EP_from_EPSfr_ff"] = df["EPSfr_ff"] / df["Price"]
df["PE_from_NI_Shares"] = df["Price"] / df["EPS_from_NI"]
df["EP_from_NI_Shares"] = df["EPS_from_NI"] / df["Price"]

# LSEG community formula: Implied_EPS ≈ ROUND(TR.EPSActValue(Period=LTM), 3)
df["Implied_EPS_LTM_formula"] = df["EPSAct_LTM"].round(3)
df["PE_LTM_formula"] = df["Price"] / df["Implied_EPS_LTM_formula"]
df["EP_LTM_formula"] = df["Implied_EPS_LTM_formula"] / df["Price"]

# Refinitivin implied EPS vertailua varten
df["Implied_EPS_REF"] = df["Price"] / df["REF_PE"]
df["Implied_EPS"] = df["Implied_EPS_REF"]

df.head()

,Date,Price,MktCap,Shares,REF_PE,EPSAct,EPSfr,NetIncome,EPSAct_LTM,PE_from_EPSAct,...,EP_from_EPSAct_ff,PE_from_EPSfr_ff,EP_from_EPSfr_ff,PE_from_NI_Shares,EP_from_NI_Shares,Implied_EPS_LTM_formula,PE_LTM_formula,EP_LTM_formula,Implied_EPS_REF,Implied_EPS
0,2024-01-02,32.4800,24983975163.8400,768199747,16.1272,3.0400,2.4600,1891000000,3.0670,10.6842,...,0.0936,13.2033,0.0757,13.1947,0.0758,3.0670,10.5902,0.0944,2.0140,2.0140
1,2024-01-03,31.8000,24460911644.4000,768199747,15.7896,3.0400,2.4600,1891000000,3.0670,10.4605,...,0.0956,12.9268,0.0774,12.9184,0.0774,3.0670,10.3684,0.0964,2.0140,2.0140
2,2024-01-04,32.2700,24822440841.6600,768199747,16.0229,3.0400,2.4600,1891000000,3.0670,10.6151,...,0.0942,13.1179,0.0762,13.1094,0.0763,3.0670,10.5217,0.0950,2.0140,2.0140
3,2024-01-05,32.4000,24922438279.1999,768199747,16.0875,3.0400,2.4600,1891000000,3.0670,10.6579,...,0.0938,13.1707,0.0759,13.1622,0.0760,3.0670,10.5641,0.0947,2.0140,2.0140
4,2024-01-08,32.2700,24822440841.6599,768199747,16.0229,3.0400,2.4600,1891000000,3.0670,10.6151,...,0.0942,13.1179,0.0762,13.1094,0.0763,3.0670,10.5217,0.0950,2.0140,2.0140


In [189]:
# Hae kvartaalisarjat ja rakenna implied EPS itse
q1 = rd.get_data(
    universe=[STOCK],
    fields=["TR.EPSActValue.periodenddate", "TR.EPSActValue"],
    parameters={"Period": "FQ-19:FQ0", "Frq": "FQ", "Curn": "EUR"}
)

q2 = rd.get_data(
    universe=[STOCK],
    fields=["TR.EPSFRActValue.periodenddate", "TR.EPSFRActValue"],
    parameters={"Period": "FQ-19:FQ0", "Frq": "FQ", "Curn": "EUR"}
)

q3 = rd.get_data(
    universe=[STOCK],
    fields=["TR.F.NetIncAfterTax.periodenddate", "TR.F.NetIncAfterTax"],
    parameters={"Period": "FQ-19:FQ0", "Frq": "FQ", "Curn": "EUR"}
)

def prepare_quarterly_series(raw_df, value_name):
    q = raw_df.copy()
    if "Instrument" in q.columns:
        q = q.drop(columns="Instrument")
    q.columns = ["QDate", value_name]
    q["QDate"] = pd.to_datetime(q["QDate"])
    q[value_name] = pd.to_numeric(q[value_name], errors="coerce")
    return q.sort_values("QDate")

q_eps_act = prepare_quarterly_series(q1, "EPSAct_Q")
q_eps_fr = prepare_quarterly_series(q2, "EPSfr_Q")
q_net_income = prepare_quarterly_series(q3, "NetIncome_Q")

q = (
    q_eps_act
    .merge(q_eps_fr, on="QDate", how="outer")
    .merge(q_net_income, on="QDate", how="outer")
    .sort_values("QDate")
)

q["TTM_EPSAct"] = q["EPSAct_Q"].rolling(4, min_periods=4).sum()
q["TTM_EPSfr"] = q["EPSfr_Q"].rolling(4, min_periods=4).sum()
q["TTM_NetIncome"] = q["NetIncome_Q"].rolling(4, min_periods=4).sum()

print("Kvartaalidata ja itse lasketut TTM-sarjat:")
print(q.to_string(index=False))

# Levita viimeisin tunnettu TTM arvo paivadataan
df = df.sort_values("Date").copy()
q_ttm = (
    q[["QDate", "TTM_EPSAct", "TTM_EPSfr", "TTM_NetIncome"]]
    .dropna(how="all", subset=["TTM_EPSAct", "TTM_EPSfr", "TTM_NetIncome"])
    .rename(columns={"QDate": "Date"})
    .sort_values("Date")
)
df = pd.merge_asof(df, q_ttm, on="Date", direction="backward")

df["TTM_EPS_from_NI"] = df["TTM_NetIncome"] / df["Shares_ff"]

# q1-pohjainen TTM vertailua varten
df["PE_from_TTM"] = df["Price"] / df["TTM_EPSAct"]
df["EP_from_TTM"] = df["TTM_EPSAct"] / df["Price"]

# Oma implied EPS: ensin EPSAct, sitten EPSfr, lopuksi NI / shares fallback
df["Implied_EPS_self"] = (
    df["TTM_EPSAct"]
    .combine_first(df["TTM_EPSfr"])
    .combine_first(df["TTM_EPS_from_NI"])
)

df["Implied_EPS_self_source"] = np.select(
    [
        df["TTM_EPSAct"].notna(),
        df["TTM_EPSAct"].isna() & df["TTM_EPSfr"].notna(),
        df["TTM_EPSAct"].isna() & df["TTM_EPSfr"].isna() & df["TTM_EPS_from_NI"].notna(),
    ],
    ["TTM_EPSAct", "TTM_EPSfr", "TTM_NetIncome_per_share"],
    default=None,
)

df["PE_self"] = df["Price"] / df["Implied_EPS_self"]
df["EP_self"] = df["Implied_EPS_self"] / df["Price"]

check = df[[
    "Date", "REF_PE", "Implied_EPS_REF", "EPSAct_LTM", "Implied_EPS_LTM_formula",
    "TTM_EPSAct", "TTM_EPSfr", "TTM_EPS_from_NI", "Implied_EPS_self",
    "Implied_EPS_self_source", "PE_LTM_formula", "PE_from_TTM", "PE_self", "EP_self"
]].copy()
check["Diff_formula_EPS"] = (check["Implied_EPS_LTM_formula"] - check["Implied_EPS_REF"]).round(6)
check["Diff_TTM_%"] = ((check["PE_from_TTM"] - check["REF_PE"]) / check["REF_PE"] * 100).round(2)
check["Diff_formula_%"] = ((check["PE_LTM_formula"] - check["REF_PE"]) / check["REF_PE"] * 100).round(2)
check["Diff_self_%"] = ((check["PE_self"] - check["REF_PE"]) / check["REF_PE"] * 100).round(2)
print("\nREF_PE vs LTM-formula vs itse laskettu EPS/PE (viimeiset 20):")
print(check.tail(20).to_string(index=False))

Kvartaalidata ja itse lasketut TTM-sarjat:
     QDate  EPSAct_Q  EPSfr_Q  NetIncome_Q  TTM_EPSAct  TTM_EPSfr   TTM_NetIncome
2021-03-31    0.3100   0.4900    375000000         NaN        NaN             NaN
2021-06-30    0.3100   0.5600    432000000         NaN        NaN             NaN
2021-09-30    0.4200   0.6600    512000000         NaN        NaN             NaN
2021-12-31    0.4900   0.5900    456000000      1.5300     2.3000 1775000000.0000
2022-03-31    0.4500   0.8300    640000000      1.6700     2.6400 2040000000.0000
2022-06-30    0.9600   0.7800    599000000      2.3200     2.8600 2207000000.0000
2022-09-30    0.8580   0.1800    139000000      2.7580     2.3800 1834000000.0000
2022-12-31    0.8400   0.6700    514000000      3.1080     2.4600 1892000000.0000
2023-03-31    0.7200   0.3100    238000000      3.3780     1.9400 1490000000.0000
2023-06-30    0.6300   0.3400    259000000      3.0480     1.5000 1150000000.0000
2023-09-30    0.8770   0.7000    539000000      3.0670 

/var/folders/x_/kykmlvrj3js3pt0rhl13sbq80000gn/T/ipykernel_74390/2217587564.py:67:FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.


In [190]:
# Trial and error: laita ehdokkaat vierekkain ja katso silmalla mikä osuu Refinitiviin
extra_field_candidates = {
    "EPSFR_LTM": "TR.EPSFRActValue(Period=LTM)",
}

for label, field in extra_field_candidates.items():
    try:
        extra = rd.get_data(
            universe=[STOCK],
            fields=["TR.PriceClose.date", field],
            parameters=PARAMS,
        )
        if len(extra.columns) == 3:
            extra.columns = ["Instrument", "Date", label]
            extra = extra.drop(columns="Instrument")
        else:
            extra.columns = ["Date", label]

        extra["Date"] = pd.to_datetime(extra["Date"])
        extra[label] = pd.to_numeric(extra[label], errors="coerce")
        df = df.merge(extra, on="Date", how="left")
        print(f"Haettu lisakandidaatti: {label}")
    except Exception as exc:
        print(f"Ei saatu kenttaa {field}: {exc}")

candidate_cols = [
    "Implied_EPS_LTM_formula",
    "EPSAct",
    "EPSfr",
    "TTM_EPSAct",
    "TTM_EPSfr",
    "TTM_EPS_from_NI",
]

if "EPSFR_LTM" in df.columns:
    candidate_cols.insert(1, "EPSFR_LTM")

trial_compare = df.loc[df["Implied_EPS_REF"].notna(), [
    "Date", "Price", "REF_PE", "Implied_EPS_REF", *candidate_cols
]].copy()

eps_cols = ["Implied_EPS_REF", *candidate_cols]
trial_compare[eps_cols] = trial_compare[eps_cols].round(3)

for col in candidate_cols:
    trial_compare[f"match_{col}"] = trial_compare[col] == trial_compare["Implied_EPS_REF"]

print("Implied EPS -vertailu, katso silmalla mikä sarake osuu Refinitivin lukuun:")
print(trial_compare.tail(40).to_string(index=False))

# Vaihda tahan se sarake, joka nayttaa osuvan parhaiten Refinitivin implied EPS:aan
TRIAL_SOURCE_COL = "EPSFR_LTM" if "EPSFR_LTM" in df.columns else "TTM_EPSfr"
print(f"\nKaytossa oleva trial-sarja: {TRIAL_SOURCE_COL}")

df["Implied_EPS_trial"] = df[TRIAL_SOURCE_COL]
df["Implied_EPS_trial_source"] = TRIAL_SOURCE_COL
df["PE_trial"] = df["Price"] / df["Implied_EPS_trial"]
df["EP_trial"] = df["Implied_EPS_trial"] / df["Price"]

print("\nTrial-sarjan viimeiset 20 riviä:")
print(df[["Date", "Implied_EPS_REF", "Implied_EPS_trial", "Implied_EPS_trial_source", "PE_trial", "EP_trial"]].tail(20).to_string(index=False))

Haettu lisakandidaatti: EPSFR_LTM
Implied EPS -vertailu, katso silmalla mikä sarake osuu Refinitivin lukuun:
      Date   Price  REF_PE  Implied_EPS_REF  Implied_EPS_LTM_formula  EPSFR_LTM  EPSAct   EPSfr  TTM_EPSAct  TTM_EPSfr  TTM_EPS_from_NI  match_Implied_EPS_LTM_formula  match_EPSFR_LTM  match_EPSAct  match_EPSfr  match_TTM_EPSAct  match_TTM_EPSfr  match_TTM_EPS_from_NI
2025-01-14 12.6050 22.0529           0.5720                   0.9840     0.5700  2.8800  1.8700      0.1940    -0.1300          -0.1240                          False            False         False        False             False            False                  False
2025-01-15 13.0600 22.8489           0.5720                   0.9840     0.5700  2.8800  1.8700      0.1940    -0.1300          -0.1240                          False            False         False        False             False            False                  False
2025-01-16 12.5450 21.9479           0.5720                   0.9840     0.5700  2.8

In [191]:
# Implied EPS -taulu
implied_eps = df[[
    "Date", "Price", "Shares", "MktCap", "REF_PE",
    "Implied_EPS_REF", "EPSAct_LTM", "Implied_EPS_LTM_formula",
    "TTM_EPSAct", "TTM_EPSfr",
    "TTM_EPS_from_NI", "Implied_EPS_self", "Implied_EPS_self_source",
    "Implied_EPS_trial", "Implied_EPS_trial_source",
    "PE_LTM_formula", "EP_LTM_formula", "PE_self", "EP_self", "PE_trial", "EP_trial"
]].copy()

implied_eps["REF_EP"] = 1 / implied_eps["REF_PE"]
implied_eps["Formula_diff_vs_REF"] = implied_eps["Implied_EPS_LTM_formula"] - implied_eps["Implied_EPS_REF"]
implied_eps["Self_diff_vs_REF"] = implied_eps["Implied_EPS_self"] - implied_eps["Implied_EPS_REF"]
implied_eps["Trial_diff_vs_REF"] = implied_eps["Implied_EPS_trial"] - implied_eps["Implied_EPS_REF"]
implied_eps["neg_eps_flag"] = (implied_eps["Implied_EPS_self"] <= 0).astype("Int64")
implied_eps.loc[implied_eps["Implied_EPS_self"].isna(), "neg_eps_flag"] = pd.NA
implied_eps["neg_eps_trial_flag"] = (implied_eps["Implied_EPS_trial"] <= 0).astype("Int64")
implied_eps.loc[implied_eps["Implied_EPS_trial"].isna(), "neg_eps_trial_flag"] = pd.NA
implied_eps["ref_pe_is_nan"] = implied_eps["REF_PE"].isna().astype(int)
implied_eps["self_available"] = implied_eps["Implied_EPS_self"].notna().astype(int)
implied_eps["ref_pe_missing_but_self_available"] = (
    implied_eps["REF_PE"].isna() & implied_eps["Implied_EPS_self"].notna()
).astype(int)

implied_eps.to_csv("implied_eps.csv", index=False)
print(f"implied_eps.csv ({len(implied_eps)} riviä)")
implied_eps

implied_eps.csv (501 riviä)


,Date,Price,Shares,MktCap,REF_PE,Implied_EPS_REF,EPSAct_LTM,Implied_EPS_LTM_formula,TTM_EPSAct,TTM_EPSfr,...,EP_trial,REF_EP,Formula_diff_vs_REF,Self_diff_vs_REF,Trial_diff_vs_REF,neg_eps_flag,neg_eps_trial_flag,ref_pe_is_nan,self_available,ref_pe_missing_but_self_available
0,2024-01-02,32.4800,768199747,24983975163.8400,16.1272,2.0140,3.0670,3.0670,2.8870,1.8700,...,0.0622,0.0620,1.0530,0.8730,0.0060,0,0,0,1,0
1,2024-01-03,31.8000,768199747,24460911644.4000,15.7896,2.0140,3.0670,3.0670,2.8870,1.8700,...,0.0635,0.0633,1.0530,0.8730,0.0060,0,0,0,1,0
2,2024-01-04,32.2700,768199747,24822440841.6600,16.0229,2.0140,3.0670,3.0670,2.8870,1.8700,...,0.0626,0.0624,1.0530,0.8730,0.0060,0,0,0,1,0
3,2024-01-05,32.4000,768199747,24922438279.1999,16.0875,2.0140,3.0670,3.0670,2.8870,1.8700,...,0.0623,0.0622,1.0530,0.8730,0.0060,0,0,0,1,0
4,2024-01-08,32.2700,768199747,24822440841.6599,16.0229,2.0140,3.0670,3.0670,2.8870,1.8700,...,0.0626,0.0624,1.0530,0.8730,0.0060,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,2025-12-19,18.5000,768243610,14230404573.0000,<NA>,<NA>,0.1560,0.1560,0.1560,-0.1400,...,-0.0076,<NA>,<NA>,<NA>,<NA>,0,1,1,1,1
497,2025-12-22,18.6850,768243610,14372708618.7300,<NA>,<NA>,0.1560,0.1560,0.1560,-0.1400,...,-0.0075,<NA>,<NA>,<NA>,<NA>,0,1,1,1,1
498,2025-12-23,18.8850,768243610,14526550830.3300,<NA>,<NA>,0.1560,0.1560,0.1560,-0.1400,...,-0.0074,<NA>,<NA>,<NA>,<NA>,0,1,1,1,1
499,2025-12-29,19.2400,768243610,14799620755.9200,<NA>,<NA>,0.1560,0.1560,0.1560,-0.1400,...,-0.0073,<NA>,<NA>,<NA>,<NA>,0,1,1,1,1


In [192]:
# PE-taulu
pe = df[["Date", "REF_PE",
         "PE_LTM_formula", "PE_trial",
         "PE_self", "PE_from_EPSAct", "PE_from_EPSfr", "PE_from_NetIncome_MktCap",
         "PE_from_EPSAct_ff", "PE_from_EPSfr_ff", "PE_from_NI_Shares",
         "PE_from_TTM", "Implied_EPS_REF", "Implied_EPS_LTM_formula", "Implied_EPS_trial", "Implied_EPS_trial_source", "Implied_EPS_self", "Implied_EPS_self_source"]].copy()
pe.to_csv("pe.csv", index=False)
print(f"pe.csv ({len(pe)} riviä)")
pe

pe.csv (501 riviä)


,Date,REF_PE,PE_LTM_formula,PE_trial,PE_self,PE_from_EPSAct,PE_from_EPSfr,PE_from_NetIncome_MktCap,PE_from_EPSAct_ff,PE_from_EPSfr_ff,PE_from_NI_Shares,PE_from_TTM,Implied_EPS_REF,Implied_EPS_LTM_formula,Implied_EPS_trial,Implied_EPS_trial_source,Implied_EPS_self,Implied_EPS_self_source
0,2024-01-02,16.1272,10.5902,16.0792,11.2504,10.6842,13.2033,13.2120,10.6842,13.2033,13.1947,11.2504,2.0140,3.0670,2.0200,EPSFR_LTM,2.8870,TTM_EPSAct
1,2024-01-03,15.7896,10.3684,15.7426,11.0149,10.4605,12.9268,12.9354,10.4605,12.9268,12.9184,11.0149,2.0140,3.0670,2.0200,EPSFR_LTM,2.8870,TTM_EPSAct
2,2024-01-04,16.0229,10.5217,15.9752,11.1777,10.6151,13.1179,13.1266,10.6151,13.1179,13.1094,11.1777,2.0140,3.0670,2.0200,EPSFR_LTM,2.8870,TTM_EPSAct
3,2024-01-05,16.0875,10.5641,16.0396,11.2227,10.6579,13.1707,13.1795,10.6579,13.1707,13.1622,11.2227,2.0140,3.0670,2.0200,EPSFR_LTM,2.8870,TTM_EPSAct
4,2024-01-08,16.0229,10.5217,15.9752,11.1777,10.6151,13.1179,13.1266,10.6151,13.1179,13.1094,11.1777,2.0140,3.0670,2.0200,EPSFR_LTM,2.8870,TTM_EPSAct
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,2025-12-19,<NA>,118.5897,-132.1429,118.5897,108.8235,-154.1667,-151.3873,108.8235,-154.1667,-151.1969,118.5897,<NA>,0.1560,-0.1400,EPSFR_LTM,0.1560,TTM_EPSAct
497,2025-12-22,<NA>,119.7756,-133.4643,119.7756,109.9118,-155.7083,-152.9012,109.9118,-155.7083,-152.7088,119.7756,<NA>,0.1560,-0.1400,EPSFR_LTM,0.1560,TTM_EPSAct
498,2025-12-23,<NA>,121.0577,-134.8929,121.0577,111.0882,-157.3750,-154.5378,111.0882,-157.3750,-154.3434,121.0577,<NA>,0.1560,-0.1400,EPSFR_LTM,0.1560,TTM_EPSAct
499,2025-12-29,<NA>,123.3333,-137.4286,123.3333,113.1765,-160.3333,-157.4428,113.1765,-160.3333,-157.2448,123.3333,<NA>,0.1560,-0.1400,EPSFR_LTM,0.1560,TTM_EPSAct


In [193]:
# EP-taulu
ep = df[["Date"]].copy()
ep["REF_EP"]              = 1 / df["REF_PE"]
ep["EP_LTM_formula"]      = df["EP_LTM_formula"]
ep["EP_trial"]            = df["EP_trial"]
ep["EP_self"]             = df["EP_self"]
ep["EP_from_EPSAct"]      = df["EP_from_EPSAct"]
ep["EP_from_EPSfr"]       = df["EP_from_EPSfr"]
ep["EP_NetIncome_MktCap"] = df["EP_NetIncome_MktCap"]
ep["EP_from_EPSAct_ff"]   = df["EP_from_EPSAct_ff"]
ep["EP_from_EPSfr_ff"]    = df["EP_from_EPSfr_ff"]
ep["EP_from_NI_Shares"]   = df["EP_from_NI_Shares"]
ep["EP_from_TTM"]         = df["EP_from_TTM"]
ep.to_csv("ep.csv", index=False)
print(f"ep.csv ({len(ep)} riviä)")
ep

ep.csv (501 riviä)


,Date,REF_EP,EP_LTM_formula,EP_trial,EP_self,EP_from_EPSAct,EP_from_EPSfr,EP_NetIncome_MktCap,EP_from_EPSAct_ff,EP_from_EPSfr_ff,EP_from_NI_Shares,EP_from_TTM
0,2024-01-02,0.0620,0.0944,0.0622,0.0889,0.0936,0.0757,0.0757,0.0936,0.0757,0.0758,0.0889
1,2024-01-03,0.0633,0.0964,0.0635,0.0908,0.0956,0.0774,0.0773,0.0956,0.0774,0.0774,0.0908
2,2024-01-04,0.0624,0.0950,0.0626,0.0895,0.0942,0.0762,0.0762,0.0942,0.0762,0.0763,0.0895
3,2024-01-05,0.0622,0.0947,0.0623,0.0891,0.0938,0.0759,0.0759,0.0938,0.0759,0.0760,0.0891
4,2024-01-08,0.0624,0.0950,0.0626,0.0895,0.0942,0.0762,0.0762,0.0942,0.0762,0.0763,0.0895
...,...,...,...,...,...,...,...,...,...,...,...,...
496,2025-12-19,<NA>,0.0084,-0.0076,0.0084,0.0092,-0.0065,-0.0066,0.0092,-0.0065,-0.0066,0.0084
497,2025-12-22,<NA>,0.0083,-0.0075,0.0083,0.0091,-0.0064,-0.0065,0.0091,-0.0064,-0.0065,0.0083
498,2025-12-23,<NA>,0.0083,-0.0074,0.0083,0.0090,-0.0064,-0.0065,0.0090,-0.0064,-0.0065,0.0083
499,2025-12-29,<NA>,0.0081,-0.0073,0.0081,0.0088,-0.0062,-0.0064,0.0088,-0.0062,-0.0064,0.0081
